In [8]:
import sys
import numpy as np
import torch 
# still do preprocessing in scipy
import scipy.sparse as sp
import anndata as ad 
from importlib import reload

import h5py
import seaborn as sns
import matplotlib.pyplot as plt

from tqdm import tqdm
import pandas as pd

# import factor model from beta-dirichlet-factor
sys.path.append('/gpfs/commons/home/kisaev/Leaflet-private/src/beta-dirichlet-factor')
import factor_model
reload(factor_model)

sns.set(style='white', context='notebook', rc={'figure.figsize':(6,6)}, font_scale=1.5)

# Append this directory to sys.path
sys.path.append('/gpfs/commons/home/kisaev/Leaflet-private/src/clustering/')
import load_cluster_data as llc 

# Append also simulation directory
sys.path.append("/gpfs/commons/home/kisaev/Leaflet-private/src/simulation/")
import simulate_counts as sim 
reload (sim)

sys.path.append("/gpfs/commons/home/kisaev/Leaflet-private/src/visualization/")
import vis as vis

2.3.0+cu121
12.1


In [3]:
adata = ad.read_h5ad('/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/Leafletall_ages_brain_intron_clusters_adata.h5ad')

/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [9]:
adata

AnnData object with n_obs × n_vars = 12657 × 48304
    obs: 'cell_id', 'age', 'batch', 'cell_ontology_class', 'mouse.id', 'sex', 'subtissue', 'tissue', 'louvain', 'leiden'
    var: 'Cluster', 'junction_id', 'gene_id'
    layers: 'Cluster_Counts', 'Junction_Counts'

In [12]:
adata.layers["Junction_Counts"], adata.layers["Cluster_Counts"]

(<12657x48304 sparse matrix of type '<class 'numpy.int64'>'
 	with 13523365 stored elements in Compressed Sparse Row format>,
 <12657x48304 sparse matrix of type '<class 'numpy.int64'>'
 	with 13523365 stored elements in Compressed Sparse Row format>)

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

float_type = { 
        "device": device, 
        "dtype": torch.float,
    }

Using device: cpu


In [ ]:
# let's write a new load_cluster_data function using AnnData objects...
def load_anndata_counts(input_file=None, num_cells_sample=None, max_intron_count=None, remove_singletons=True, has_genes="no"):
    